In [19]:
!pip install -q FlagEmbedding qdrant-client tqdm

In [20]:

import json
import logging
import os
import re
import uuid
from pathlib import Path

from tqdm import tqdm
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient, models

In [21]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [22]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DRIVE_ROOT = Path("/content/drive/MyDrive/RagApplication/oracle_rag")
PARSED_DIR = DRIVE_ROOT / "parsed"
LOGS_DIR = DRIVE_ROOT / "logs"
EMBED_PROGRESS_FILE = DRIVE_ROOT / "embed_progress.json"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
def _secret(name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ[name]


QDRANT_URL = _secret("QDRANT_URL")
QDRANT_API_KEY = _secret("QDRANT_API_KEY")
COLLECTION_NAME = "oracle_docs_321"

DENSE_DIM = 1024
EMBED_BATCH_SIZE = 16
UPSERT_BATCH_SIZE = 128
EMBED_MAX_LENGTH = 2048          # tokens; units are far smaller than this

TARGET_CHARS = 1800              # flush a text unit once it reaches this size
MAX_TABLE_CHARS = 3000           # split bigger tables by rows (header repeated)
MAX_EMBED_CHARS = 8000           # hard cap on text sent to the embedder
LEAD_IN_MAX_CHARS = 300          # short paragraph before a table/code block

logging.basicConfig(
    filename=LOGS_DIR / "embedding.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,                  # Colab already has a root handler otherwise
)
log = logging.getLogger("embed")

In [24]:
def load_progress() -> dict:
    if EMBED_PROGRESS_FILE.exists():
        return json.loads(EMBED_PROGRESS_FILE.read_text())
    return {}


def save_progress(progress: dict):
    EMBED_PROGRESS_FILE.write_text(json.dumps(progress, indent=2))


NAMESPACE = uuid.UUID("12345678-1234-5678-1234-567812345678")


def unit_point_id(doc_name: str, unit_index: int) -> str:
    return str(uuid.uuid5(NAMESPACE, f"{doc_name}_u{unit_index}"))

In [25]:
NOISE_RE = re.compile(r"^(page\s+)?\d+(\s+of\s+\d+)?$", re.I)   # "12", "Page 12"


def split_table(md: str, max_chars: int) -> list[str]:
    """Split a markdown table by rows, repeating header + separator rows."""
    lines = md.splitlines()
    if len(md) <= max_chars or len(lines) < 4:
        return [md]
    header, rows = lines[:2], lines[2:]
    parts, cur, size = [], [], 0
    for r in rows:
        if cur and size + len(r) > max_chars:
            parts.append("\n".join(header + cur))
            cur, size = [], 0
        cur.append(r)
        size += len(r) + 1
    if cur:
        parts.append("\n".join(header + cur))
    return parts


def build_units(raw_chunks: list[dict]) -> list[dict]:
    """Merge consecutive text chunks; keep tables / code / json / images as
    their own units. Returns dicts: type, content, embed, page_start,
    page_end, extra."""
    units: list[dict] = []
    buf: list[dict] = []
    buf_len = 0

    def flush_text():
        nonlocal buf, buf_len
        if not buf:
            return
        content = "\n".join(c["content"] for c in buf)
        units.append({
            "type": "text",
            "content": content,
            "embed": content,
            "page_start": buf[0].get("page"),
            "page_end": buf[-1].get("page"),
            "extra": {},
        })
        buf, buf_len = [], 0

    for c in raw_chunks:
        ctype = c.get("type")
        content = (c.get("content") or "").strip()
        if not content:
            continue

        # parser labelled bare numbers / "true" / "null" as json - not real json
        if ctype == "json" and not content.startswith(("{", "[")):
            ctype = "text"

        if ctype == "text":
            if NOISE_RE.match(content):
                continue
            buf.append({"content": content, "page": c.get("page")})
            buf_len += len(content) + 1
            if buf_len >= TARGET_CHARS:
                flush_text()
            continue

        # non-text block: close the running text unit first to keep order
        lead_in = ""
        if buf and len(buf[-1]["content"]) <= LEAD_IN_MAX_CHARS:
            lead_in = buf[-1]["content"]
        flush_text()

        page = c.get("page")

        if ctype == "table":
            caption = (c.get("caption") or "").strip()
            for part in split_table(content, MAX_TABLE_CHARS):
                embed = "\n".join(x for x in (lead_in, caption, part) if x)
                units.append({
                    "type": "table", "content": part, "embed": embed,
                    "page_start": page, "page_end": page,
                    "extra": {"caption": caption},
                })
        elif ctype in ("code", "json"):
            embed = "\n".join(x for x in (lead_in, content) if x)
            units.append({
                "type": ctype, "content": content, "embed": embed,
                "page_start": page, "page_end": page,
                "extra": {"language": c.get("language")},
            })
        elif ctype == "image":
            units.append({
                "type": "image", "content": content, "embed": content,
                "page_start": page, "page_end": page,
                "extra": {
                    "image_path": c.get("image_path"),
                    "figure_caption": c.get("figure_caption"),
                },
            })
        else:  # unknown type: treat as text-like standalone
            units.append({
                "type": ctype or "text", "content": content, "embed": content,
                "page_start": page, "page_end": page, "extra": {},
            })

    flush_text()
    return units

In [26]:
def load_doc_units(parsed_path: Path) -> list[dict]:
    doc_name = parsed_path.stem
    raw_chunks = json.loads(parsed_path.read_text())
    out = []
    for idx, u in enumerate(build_units(raw_chunks)):
        payload = {
            "doc": doc_name,
            "unit_index": idx,
            "type": u["type"],
            "page": u["page_start"],
            "page_end": u["page_end"],
            "content": u["content"],
        }
        payload.update({k: v for k, v in u["extra"].items() if v})
        out.append({
            "id": unit_point_id(doc_name, idx),
            "text": u["embed"][:MAX_EMBED_CHARS],
            "payload": payload,
        })
    return out


# ============================================================
# 4. SPARSE VECTOR CONVERSION (BGE-M3 lexical weights)
# ============================================================

def sparse_to_qdrant(lexical_weights: dict) -> models.SparseVector:
    return models.SparseVector(
        indices=[int(k) for k in lexical_weights.keys()],
        values=[float(v) for v in lexical_weights.values()],
    )

In [27]:
def ensure_collection(client: QdrantClient):
    if not client.collection_exists(COLLECTION_NAME):
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config={
                "dense": models.VectorParams(
                    size=DENSE_DIM, distance=models.Distance.COSINE
                )
            },
            sparse_vectors_config={"sparse": models.SparseVectorParams()},
        )
        print(f"Created collection '{COLLECTION_NAME}'")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists")

    # indexes make filtering (doc / type / neighbour lookup) fast
    for field, schema in [
        ("doc", models.PayloadSchemaType.KEYWORD),
        ("type", models.PayloadSchemaType.KEYWORD),
        ("unit_index", models.PayloadSchemaType.INTEGER),
        ("page", models.PayloadSchemaType.INTEGER),
    ]:
        try:
            client.create_payload_index(
                collection_name=COLLECTION_NAME,
                field_name=field,
                field_schema=schema,
            )
        except Exception as e:  # already exists is fine
            log.info(f"payload index {field}: {e}")


In [28]:
def embed_and_index_doc(parsed_path, model, client, progress):
    doc_name = parsed_path.stem
    try:
        print(f"\nProcessing: {doc_name}")
        units = load_doc_units(parsed_path)

        if not units:
            progress[doc_name] = "done_empty"
            save_progress(progress)
            log.info(f"{doc_name}: no embeddable units")
            return

        points = []
        for start in range(0, len(units), EMBED_BATCH_SIZE):
            batch = units[start:start + EMBED_BATCH_SIZE]
            out = model.encode(
                [u["text"] for u in batch],
                batch_size=EMBED_BATCH_SIZE,
                max_length=EMBED_MAX_LENGTH,
                return_dense=True,
                return_sparse=True,
                return_colbert_vecs=False,
            )
            for u, dense_vec, sparse_w in zip(
                batch, out["dense_vecs"], out["lexical_weights"]
            ):
                vector = {"dense": dense_vec.tolist()}
                if sparse_w:                       # skip empty sparse vectors
                    vector["sparse"] = sparse_to_qdrant(sparse_w)
                points.append(
                    models.PointStruct(id=u["id"], vector=vector, payload=u["payload"])
                )

        # drop this doc's old points (covers changed chunking), then upload
        client.delete(
            collection_name=COLLECTION_NAME,
            points_selector=models.FilterSelector(
                filter=models.Filter(must=[
                    models.FieldCondition(
                        key="doc", match=models.MatchValue(value=doc_name)
                    )
                ])
            ),
            wait=True,
        )
        for s in range(0, len(points), UPSERT_BATCH_SIZE):
            client.upsert(
                collection_name=COLLECTION_NAME,
                points=points[s:s + UPSERT_BATCH_SIZE],
                wait=True,
            )

        progress[doc_name] = "done"
        log.info(f"{doc_name}: indexed {len(points)} units")
        print(f"OK {doc_name}: {len(points)} units indexed")

    except Exception as e:
        progress[doc_name] = "failed"
        log.exception(f"{doc_name} FAILED")
        print(f"FAILED {doc_name}: {e}")

    save_progress(progress)


# ============================================================
# 7. BATCH RUNNER
# ============================================================

def run_batch():
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    print("Qdrant collections:", client.get_collections())
    ensure_collection(client)

    existing_points = client.count(collection_name=COLLECTION_NAME, exact=True).count
    progress = load_progress()
    already_done = sum(1 for v in progress.values() if v.startswith("done"))
    print(f"Collection currently has {existing_points} points.")
    print(f"Progress file shows {already_done} docs already done.")
    if existing_points > 0 and already_done == 0:
        input("Collection has data but progress file is empty. Press Enter to continue, or stop the cell now.")

    import torch
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=torch.cuda.is_available())

    parsed_files = sorted(PARSED_DIR.glob("*.json"))
    if not parsed_files:
        print(f"No parsed docs found in {PARSED_DIR}")
        return

    pending = [p for p in parsed_files
               if progress.get(p.stem) not in ("done", "done_empty")]
    print(f"{len(parsed_files)} parsed docs, {len(pending)} pending.")

    for p in tqdm(pending, desc="Embedding + uploading"):
        embed_and_index_doc(p, model, client, progress)

    names = {p.stem for p in parsed_files}
    done = sum(1 for k, v in progress.items() if k in names and v.startswith("done"))
    failed = sum(1 for k, v in progress.items() if k in names and v == "failed")
    total = client.count(collection_name=COLLECTION_NAME, exact=True).count
    print(f"\nFINISHED: {done} docs indexed, {failed} failed, {total} points in '{COLLECTION_NAME}'")

In [29]:
run_batch()

Qdrant collections: collections=[CollectionDescription(name='oracle_docs_321')]
Collection 'oracle_docs_321' already exists
Collection currently has 17914 points.
Progress file shows 48 docs already done.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

18 parsed docs, 1 pending.


Embedding + uploading:   0%|          | 0/1 [00:00<?, ?it/s]


Processing: Comprehensive_Guide_RAG_Optimized_Version1



pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 55.33it/s]

Embedding + uploading: 100%|██████████| 1/1 [00:04<00:00,  4.81s/it]

OK Comprehensive_Guide_RAG_Optimized_Version1: 22 units indexed

FINISHED: 18 docs indexed, 0 failed, 17936 points in 'oracle_docs_321'
